requires xtb to be downloaded :

easiest way is through 
```
conda install xtb 
``` 

In [1]:
from arcskin.nanoreactor_run import SMILESPack 
from arcskin.nanoreactor_run import XTBNanoReactor

In [2]:
compounds = {
    "NO2":  "[N+](=O)[O-]",
    "NO":   "[N]=O",
    "O2":   "O=O",
}

combination = ["NO","O2"] # can run multiple combinations 
sp = SMILESPack(["[N]=O","O=O"])
cells, charges = sp.run_multiple(2,counts=[4,4],min_dist=2.2) # 4 of each molecule, with a minimum distance of 2\AA
cells

[Atoms(symbols='NONONONO9', pbc=False), Atoms(symbols='NONONONO9', pbc=False)]

In [3]:
xnr = XTBNanoReactor(combination={"NO":"[N]=O","O2":"O=O"},cells=cells,charges=charges,time=2)
xnr.run(location="./nanoreactor_run_directories/")

In [4]:
xnr.trajectory_locations

['./nanoreactor_run_directories/seed_0/xtb.trj',
 './nanoreactor_run_directories/seed_1/xtb.trj']

In [5]:
## Analyse the nanoreactor output 

In [6]:
from arcskin.nanoreactor_analysis import AnalyseNanoReactor 
anr = AnalyseNanoReactor(skip_step=50,calculator=None)
anr.read_trajectory(xnr.trajectory_locations,format="xyz")
anr.analyse_trajectories()

Warp 1.18.0 initialized:
   CUDA not enabled in this build
   Devices:
     "cpu"      : "arm"
   Kernel cache:
     /Users/badw/Library/Caches/warp/1.18.0
calculating with default calculator: aimnet
reactants found = ['[N][O]', 'O=O']


100%|██████████| 8/8 [00:19<00:00,  2.47s/it]


In [7]:
import pandas as pd 
pd.DataFrame(anr.unique_smiles_counter)

,0,1
[N]=O,12.0,11
[O][O],1.0,1
[O]N(=O)=O,1.0,1
[O]N=O,1.0,1
[O]O[N][O],NaN,1
[O]O[O],NaN,4


In [14]:
anr.get_sensible_smiles()
possible_paths = anr.get_possible_paths_dict()
possible_paths

{0: {'reactants': ['[N][O]', 'O=O'], 'products': ['[N]=O', '[O][O]']},
 1: {'reactants': ['[N][O]', 'O=O'], 'products': ['[O]N(=O)=O']},
 2: {'reactants': ['[N][O]', 'O=O'], 'products': ['[O]O[N][O]']}}

In [ ]:
from arcskin.nanoreactor_analysis import calculate_molecular_similarity 
import itertools as it 
possible_products = list(it.chain(*[possible_paths[i]["products"] for i in possible_paths]))
possible_products
reactants = possible_paths[0]["reactants"]
calculate_molecular_similarity(possible_products[2],possible_products[3])

[14:42:25] DEPRECATION WARNING: please use MorganGenerator
[14:42:25] DEPRECATION WARNING: please use MorganGenerator


0.07142857142857142

In [ ]:
from arcskin.nanoreactor_run import smiles_to_atoms
from ase.visualize import view 

view(
        smiles_to_atoms("[O][O]")[0],
        viewer="x3d"
        )

In [23]:
view(
        smiles_to_atoms("O=O")[0],
        viewer="x3d"
        )